# Byte Pair Encoding (BPE)

**Audience:** complete beginners. You do not need prior NLP.

BPE is the algorithm behind GPT-style tokenizers (and many Llama tokenizers). It starts from
tiny pieces (characters, or even bytes) and **glues the most common adjacent pair** over and
over until the vocabulary is big enough.

After this notebook you will be able to:

1. Explain why word-level vocabularies break on new words.
2. Train a tiny BPE model by hand and watch each merge.
3. Encode a new word with the merge list you learned.
4. Train the same idea with HuggingFace `tokenizers` (current API).
5. Inspect a production BPE vocabulary with `tiktoken`.


## Learning path

```mermaid
flowchart LR
  problem[OOV problem] --> scratch[From-scratch BPE]
  scratch --> encode[Encode with merges]
  encode --> hf[HuggingFace BpeTrainer]
  hf --> tik[tiktoken production BPE]
```


## Why BPE exists

A **word-level** tokenizer stores every whole word. The moment a student types `unhappiness`
and that string never appeared in training, the model sees `[UNK]`. Spelling information is
gone.

BPE's bet: keep **frequent** words intact, and build rare words from **reusable pieces**.
`low` + `est` can form `lowest` even if `lowest` itself was rare.

The original NLP paper is Sennrich, Haddow, and Birch (2016). Neural nets had already used a
similar compression trick (Gage, 1994). GPT-2 later ran BPE on **UTF-8 bytes** so nothing is
unknown.


## The algorithm (picture)

```mermaid
flowchart TD
  corpus[Corpus of words plus counts] --> split[Split each word into characters]
  split --> endmark["Append an end-of-word mark </w>"]
  endmark --> count[Count every adjacent pair]
  count --> pick[Pick the pair with the highest count]
  pick --> merge[Glue that pair into one new symbol]
  merge --> vocab[Add the new symbol to the vocabulary]
  vocab --> enough{Reached N merges?}
  enough -->|no| count
  enough -->|yes| done[Save merge list and vocab]
```

`</w>` matters. Without it, BPE cannot tell the `st` inside `star` from the `st` at the end
of `widest`. The end mark is a boundary.


## Setup — paths and corpus


In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets

def find_root() -> Path:
    here = Path.cwd()
    for candidate in [here, here.parent]:
        if (candidate / "data" / "tiny_corpus.txt").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from the repo root or the notebooks/ folder.")

ROOT = find_root()
CORPUS = ROOT / "data" / "tiny_corpus.txt"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print(f"corpus: {CORPUS}")


corpus: /Users/sourangshupal/Downloads/tokenization-explainer/data/tiny_corpus.txt


In [2]:
text = CORPUS.read_text(encoding="utf-8")
print("First 20 lines of the toy corpus:\n")
print("\n".join(text.splitlines()[:20]))


First 20 lines of the toy corpus:

low
low
low
low
low
lower
lower
newest
newest
newest
widest
widest
the cat sat on the mat
the cat sat on the mat
tokenization is the first step of every language model
unhappiness is not a word you see every day
playing players played
walking walks walked walker
low lower lowest
new newer newest


## From scratch: count words, then characters

We only keep alphabetic tokens so the first merges stay readable. Real BPE also sees
punctuation and digits. Production GPT BPE sees raw bytes.


In [3]:
def load_word_counts(path: Path) -> Counter[str]:
    counts: Counter[str] = Counter()
    for line in path.read_text(encoding="utf-8").splitlines():
        for raw in line.lower().split():
            word = "".join(ch for ch in raw if ch.isalpha())
            if word:
                counts[word] += 1
    return counts


word_counts = load_word_counts(CORPUS)
print(f"{len(word_counts)} word types, {sum(word_counts.values())} tokens")
print("Most common:", word_counts.most_common(8))


105 word types, 152 tokens
Most common: [('the', 8), ('low', 6), ('newest', 4), ('is', 4), ('lower', 3), ('widest', 3), ('on', 3), ('not', 3)]


In [4]:
def initial_splits(counts: Counter[str]) -> dict[str, list[str]]:
    return {word: list(word) + ["</w>"] for word in counts}


splits = initial_splits(word_counts)
for word in ["low", "lower", "newest", "widest"]:
    if word in splits:
        print(f"{word:8}  {splits[word]}   x{word_counts[word]}")


low       ['l', 'o', 'w', '</w>']   x6
lower     ['l', 'o', 'w', 'e', 'r', '</w>']   x3
newest    ['n', 'e', 'w', 'e', 's', 't', '</w>']   x4
widest    ['w', 'i', 'd', 'e', 's', 't', '</w>']   x3


## Count pairs and merge the winner

A **pair** is two neighbouring symbols inside a word. If `low` appears 5 times as
`['l','o','w','</w>']`, the pair `('l','o')` gets +5, not +1. Frequency is corpus count, not
type count.


In [5]:
def pair_counts(splits: dict[str, list[str]], counts: Counter[str]) -> Counter[tuple[str, str]]:
    pairs: Counter[tuple[str, str]] = Counter()
    for word, freq in counts.items():
        symbols = splits[word]
        for left, right in zip(symbols, symbols[1:]):
            pairs[(left, right)] += freq
    return pairs


def apply_merge(splits: dict[str, list[str]], pair: tuple[str, str]) -> dict[str, list[str]]:
    a, b = pair
    glued = a + b
    updated: dict[str, list[str]] = {}
    for word, symbols in splits.items():
        out: list[str] = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                out.append(glued)
                i += 2
            else:
                out.append(symbols[i])
                i += 1
        updated[word] = out
    return updated


stats = pair_counts(splits, word_counts)
print("Top 10 pairs before any merge:")
for pair, freq in stats.most_common(10):
    print(f"  {pair!s:30} {freq}")


Top 10 pairs before any merge:
  ('s', '</w>')                  31
  ('t', '</w>')                  28
  ('e', '</w>')                  24
  ('s', 't')                     17
  ('e', 's')                     15
  ('e', 'r')                     14
  ('t', 'h')                     13
  ('e', 'n')                     13
  ('l', 'o')                     12
  ('n', 'e')                     12


## Train: repeat the merge N times

Each round we record `(left, right) → left+right`. That ordered list **is** the tokenizer.
Encoding a new word later means replaying these merges in the same order.


In [6]:
def train_bpe(
    counts: Counter[str],
    num_merges: int,
) -> tuple[list[tuple[str, str]], dict[str, list[str]]]:
    splits = initial_splits(counts)
    merges: list[tuple[str, str]] = []
    for step in range(1, num_merges + 1):
        stats = pair_counts(splits, counts)
        if not stats:
            break
        pair, freq = stats.most_common(1)[0]
        if freq < 2:
            print(f"stop at step {step}: best pair only appears {freq} time(s)")
            break
        merges.append(pair)
        splits = apply_merge(splits, pair)
        a, b = pair
        print(f"{step:02d}. merge {a!r} + {b!r}  →  {a + b!r}   (freq {freq})")
    return merges, splits


merges, trained_splits = train_bpe(word_counts, num_merges=25)
print(f"\nlearned {len(merges)} merges")
print("\nHow the classic words look after training:")
for word in ["low", "lower", "newest", "widest", "lowest"]:
    if word in trained_splits:
        print(f"  {word:8} {trained_splits[word]}")


01. merge 's' + '</w>'  →  's</w>'   (freq 31)
02. merge 't' + '</w>'  →  't</w>'   (freq 28)
03. merge 'e' + '</w>'  →  'e</w>'   (freq 24)
04. merge 'e' + 'r'  →  'er'   (freq 14)
05. merge 's' + 't</w>'  →  'st</w>'   (freq 13)
06. merge 't' + 'h'  →  'th'   (freq 13)
07. merge 'e' + 'n'  →  'en'   (freq 13)
08. merge 'l' + 'o'  →  'lo'   (freq 12)
09. merge 'n' + 'e'  →  'ne'   (freq 11)
10. merge 'o' + 'r'  →  'or'   (freq 11)
11. merge 'lo' + 'w'  →  'low'   (freq 10)
12. merge 'e' + 'st</w>'  →  'est</w>'   (freq 10)
13. merge 'w' + 'or'  →  'wor'   (freq 10)
14. merge 'l' + 'a'  →  'la'   (freq 9)
15. merge 'd' + '</w>'  →  'd</w>'   (freq 9)
16. merge 'er' + '</w>'  →  'er</w>'   (freq 8)
17. merge 'th' + 'e</w>'  →  'the</w>'   (freq 8)
18. merge 'n' + '</w>'  →  'n</w>'   (freq 8)
19. merge 'y' + '</w>'  →  'y</w>'   (freq 8)
20. merge 'w' + 'i'  →  'wi'   (freq 7)
21. merge 'c' + 'a'  →  'ca'   (freq 7)
22. merge 'n' + 'g'  →  'ng'   (freq 7)
23. merge 'low' + '</w>'  →  'l

## Encode a new word

Training never saw every possible word. Encoding still works: split into characters, then
apply **the same merges in the same order**. If `e` + `s` was merged during training, it
will merge here too.


In [7]:
def encode_word(word: str, merges: list[tuple[str, str]]) -> list[str]:
    symbols = list(word.lower()) + ["</w>"]
    for pair in merges:
        symbols = apply_merge({"w": symbols}, pair)["w"]
    return symbols


def encode_text(text: str, merges: list[tuple[str, str]]) -> list[str]:
    pieces: list[str] = []
    for raw in text.split():
        word = "".join(ch for ch in raw.lower() if ch.isalpha())
        if word:
            pieces.extend(encode_word(word, merges))
        else:
            pieces.append(raw)
    return pieces


for sample in ["lowest", "newer", "tokenization", "unhappiness"]:
    print(f"{sample:15} → {encode_word(sample, merges)}")


lowest          → ['low', 'est</w>']
newer           → ['new', 'er</w>']
tokenization    → ['to', 'k', 'en', 'i', 'z', 'a', 't', 'i', 'o', 'n</w>']
unhappiness     → ['u', 'n', 'h', 'a', 'p', 'p', 'i', 'ne', 's', 's</w>']


## Interactive playground

Type a phrase. The encoder uses **your** merge list from the cell above. Re-run training
with a different `num_merges` and this widget will still use whatever `merges` is in memory.


In [8]:
box = widgets.Text(
    value="lowest newest unhappiness",
    description="Text:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "50px"},
)
out = widgets.Output()


def _run(_change=None) -> None:
    with out:
        out.clear_output()
        pieces = encode_text(box.value, merges)
        print("pieces:", pieces)
        print("count: ", len(pieces))


box.observe(_run, names="value")
_run()
display(box, out)


Text(value='lowest newest unhappiness', description='Text:', layout=Layout(width='90%'), style=TextStyle(descr…

Output()

## Production library: HuggingFace `tokenizers`

The Rust library `tokenizers` (we installed **0.23.x** via uv) trains BPE at production
speed. The ideas are the same: a model, a pre-tokenizer, a trainer, then `encode`.

Current API — do not use old `BertTokenizer` constructors from `transformers` here. We stay
on the `tokenizers` package.


In [9]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

hf_bpe = Tokenizer(BPE(unk_token="[UNK]"))
hf_bpe.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]"],
    vocab_size=120,
    min_frequency=1,
    show_progress=False,
)
hf_bpe.train([str(CORPUS)], trainer)

encoded = hf_bpe.encode("lower newest unhappiness tokenization")
print("tokens:", encoded.tokens)
print("ids:   ", encoded.ids)
print("vocab size:", hf_bpe.get_vocab_size())


tokens: ['lower', 'newest', 'un', 'hap', 'pi', 'nes', 's', 'tokeniz', 'at', 'i', 'on']
ids:    [86, 67, 84, 100, 54, 85, 22, 90, 40, 12, 62]
vocab size: 120


In [10]:
hf_path = ARTIFACTS / "hf_bpe.json"
hf_bpe.save(str(hf_path))
reloaded = Tokenizer.from_file(str(hf_path))
print("reloaded:", reloaded.encode("widest lower").tokens)


reloaded: ['widest', 'lower']


## Production BPE: OpenAI `tiktoken`

You do not train `tiktoken`. OpenAI already ran byte-level BPE on a huge corpus and shipped
the merge table. `cl100k_base` is the GPT-4 / GPT-3.5 family. `o200k_base` is the newer
GPT-4o family. Notice how one English word is often **one** token, while a rare or
non-English string becomes several.

This is still BPE. The difference is scale and the byte-level base vocabulary.


In [11]:
import tiktoken

for name in ["cl100k_base", "o200k_base"]:
    enc = tiktoken.get_encoding(name)
    samples = [
        "tokenization",
        "unhappiness",
        "lowest newest",
        "বাংলা",
        "👋",
    ]
    print(f"\n=== {name}  (vocab ~ {enc.n_vocab}) ===")
    for s in samples:
        ids = enc.encode(s)
        pieces = [enc.decode([i]) for i in ids]
        print(f"  {s!r:20} ids={ids}  pieces={pieces!r}")



=== cl100k_base  (vocab ~ 100277) ===
  'tokenization'       ids=[5963, 2065]  pieces=['token', 'ization']
  'unhappiness'        ids=[359, 71, 67391]  pieces=['un', 'h', 'appiness']
  'lowest newest'      ids=[90998, 24519]  pieces=['lowest', ' newest']
  'বাংলা'              ids=[11372, 105, 50228, 224, 11372, 110, 42412]  pieces=['�', '�', 'া�', '�', '�', '�', 'া']
  '👋'                  ids=[9468, 239, 233]  pieces=['�', '�', '�']



=== o200k_base  (vocab ~ 200019) ===

  'tokenization'       ids=[10346, 2860]  pieces=['token', 'ization']
  'unhappiness'        ids=[373, 71, 117779]  pieces=['un', 'h', 'appiness']
  'lowest newest'      ids=[183722, 29442]  pieces=['lowest', ' newest']
  'বাংলা'              ids=[168033, 797]  pieces=['বাংল', 'া']
  '👋'                  ids=[28823, 233]  pieces=['�', '�']


## Where you will see BPE

| System | Flavour |
|---|---|
| GPT-2 / GPT-3 / GPT-4 / GPT-4o | byte-level BPE (`tiktoken`) |
| Many Llama / Mistral tokenizers | BPE (SentencePiece or HuggingFace) |
| The mini lab on the website | character BPE with `</w>` |

BPE never asks “is this a linguistically nice morpheme?” It only asks “did this pair occur a
lot?” WordPiece (next notebook) changes that scoring rule.


## Exercises

1. Re-run `train_bpe` with `num_merges=5` and `num_merges=40`. Encode `lowest`. What changed?
2. Why does `</w>` exist? Try a thought experiment: merge `st` in both `star` and `widest`.
3. HuggingFace BPE used a `Whitespace` pre-tokenizer. What would break if you skipped it?
4. Using `tiktoken`, encode your name in English and in another script. Compare token counts.

Write answers in a new cell below. There are no hidden solutions in this notebook — talk
them through in class.
